# 440 · 어댑터 저장·병합과 모델 카드

**3일차 7교시** · 슬라이드 106–110 · **GPU 필요**

## 이 노트북에서 하는 일
1. 어댑터만 저장 vs 병합 저장의 크기·운영 특성을 비교한다
2. **4bit 베이스에 학습한 어댑터를 16비트 베이스에 병합**할 때의 불일치를 확인한다
3. **병합본으로 `430`의 평가를 다시 돌린다** — 슬라이드 90의 6번(merge 후 성능 열화) 회수
4. 모델 카드를 자동 생성한다

> **주의.** 병합 후 성능이 거의 같게 나오는 경우도 많다. "열화가 관측되지 않았다"도
> 정당한 결과다. 결과를 유도하지 말고 **확인했다는 사실**을 산출물로 남기십시오.

In [ ]:
# Colab에서 처음 실행할 때만 INSTALL=True 로 바꾸고 1회 실행한다.
# 로컬(uv)에서는 requirements.txt로 이미 설치되어 있다.
# 설치 후 런타임 재시작이 필요할 수 있다.
INSTALL = False

PKGS = ("transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 "
        "peft==0.12.0 trl==0.9.6 bitsandbytes==0.43.3 tiktoken==0.7.0").split()

if INSTALL:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)
    print("설치 완료. 런타임 재시작이 필요할 수 있습니다.")

In [ ]:
# --- 저장소 루트를 import 경로에 추가 (Colab / 로컬 공통) ---
import sys, os
from pathlib import Path
for p in [Path.cwd(), *Path.cwd().parents[:3]]:
    if (p / "common" / "config.py").exists():
        sys.path.insert(0, str(p)); os.chdir(p); break
print("저장소 루트:", Path.cwd())

from common import config as C, env, artifacts as art
env.set_seed(C.SEED)

In [ ]:
import json, torch, time
import pandas as pd
from common import metrics as M, regression as REG, chat

info = env.print_env()
RUN_ID = art.latest_run()
assert RUN_ID, "artifacts/runs/ 가 비어 있습니다. 420을 먼저 실행하십시오."
assert info.cuda_available, "GPU 런타임이 필요합니다."

run_cfg = art.load_json(art.run_dir(RUN_ID) / "config.json")
eval_summary = art.load_json(art.eval_dir(RUN_ID) / "summary.json", default=None)
assert eval_summary, "430을 먼저 실행하십시오 (병합 전 지표가 필요합니다)."
print("run_id:", RUN_ID)
print("병합 전 지표:", json.dumps(eval_summary["conditions"]["tuned"]["format"], ensure_ascii=False))

---
## 1. 어댑터만 저장 vs 병합 저장

| 방식 | 크기 | 운영 |
|---|---|---|
| 어댑터만 | 수십 MB | 베이스 1개 + 어댑터 N개를 갈아 끼운다. 태스크별 모델 관리에 유리 |
| 병합 후 | 수 GB | 단일 모델 배포. 추론 시 어댑터 연산이 없어 약간 빠르다 |

In [ ]:
adapter_dir = art.run_dir(RUN_ID) / "adapter"
adapter_mb = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / 1024**2

n_params = run_cfg["lora"]["trainable_params"]
base_params = None
print(f"어댑터 크기        : {adapter_mb:.1f} MB")
print(f"학습 파라미터 수   : {n_params:,} ({run_cfg['lora']['trainable_pct']}%)")
print(f"병합본 예상 크기   : 베이스 모델 전체 크기와 같다 (fp16 기준 수 GB)")
print(f"\n같은 베이스에 어댑터 10개를 두면 {adapter_mb*10:.0f} MB,")
print(f"병합본 10개를 두면 베이스 크기 x 10 이 된다. 이것이 PEFT의 실무 이점이다.")

---
## 2. 병합의 함정 (§3.13-2)

학습은 **4bit 베이스** 위에서 했다. 병합할 때는 **16비트 베이스**에 어댑터를 더한다.
즉 학습 시 모델과 병합 후 모델이 정확히 같지 않다.

```
학습 시   : dequantize(W_4bit) + BA   <- 양자화 오차가 섞인 W 위에서 최적화됨
병합 후   : W_fp16 + BA               <- 오차 없는 W 에 같은 BA 를 더한다
```

이 불일치가 성능 변화를 만들 수 있다. **그래서 병합 후 같은 평가를 다시 돌려야 한다.**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

DTYPE = env.pick_dtype()
tok = AutoTokenizer.from_pretrained(C.MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# 16비트 베이스에 어댑터를 얹어 병합한다 (4bit가 아님에 주의)
with env.vram_probe("16비트 베이스 로드") as p:
    base16 = AutoModelForCausalLM.from_pretrained(
        C.MODEL_ID, torch_dtype=DTYPE, device_map={"": 0},
        attn_implementation=info.attn_implementation)

peft_model = PeftModel.from_pretrained(base16, str(adapter_dir))
print("\n병합 전:", type(peft_model).__name__)

merged = peft_model.merge_and_unload()
merged.eval(); merged.config.use_cache = True
print("병합 후:", type(merged).__name__)
print("어댑터 모듈이 사라지고 Linear 가중치에 흡수되었다.")

In [ ]:
# 병합이 실제로 가중치를 바꾸었는지 확인
import torch
base_ref = AutoModelForCausalLM.from_pretrained(C.MODEL_ID, torch_dtype=DTYPE)
name_check = None
for n, _ in merged.named_parameters():
    if n.endswith("q_proj.weight"):
        name_check = n; break

if name_check:
    w_merged = dict(merged.named_parameters())[name_check].detach().float().cpu()
    w_base = dict(base_ref.named_parameters())[name_check].detach().float().cpu()
    diff = (w_merged - w_base).abs()
    print(f"검사 대상: {name_check}")
    print(f"  최대 변화량 : {diff.max().item():.6f}")
    print(f"  평균 변화량 : {diff.mean().item():.6f}")
    print(f"  변화한 원소 : {100*(diff > 1e-6).float().mean().item():.1f}%")
    print("\n0이 아니면 병합이 실제로 일어난 것이다.")
del base_ref; torch.cuda.empty_cache()

---
## 3. 병합본으로 같은 평가 재실행

**`430`과 동일한 프로토콜**을 쓴다. 다른 설정으로 재면 비교가 아니다.

In [ ]:
eval_rows = art.load_jsonl(art.data_path("eval.jsonl"))
reg_items = art.load_jsonl(art.ROOT / "data" / "regression_set.jsonl")

def generate_merged(rows, gen_kwargs=None, use_system=True, desc=""):
    gk = {k: v for k, v in dict(gen_kwargs or C.GEN_EVAL).items() if v is not None}
    outs, t0 = [], time.perf_counter()
    for i, r in enumerate(rows):
        msgs = chat.build_messages(r["question"],
                                   system=(r.get("system") if use_system else None),
                                   context=r.get("context"))
        prompt = chat.render_prompt(tok, msgs)
        ins = tok(prompt, return_tensors="pt").to(merged.device)
        with torch.no_grad():
            o = merged.generate(**ins, pad_token_id=tok.pad_token_id, **gk)
        outs.append(tok.decode(o[0][ins["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        if (i + 1) % 25 == 0:
            print(f"    {desc} {i+1}/{len(rows)} ({time.perf_counter()-t0:.0f}초)")
    return outs, round(time.perf_counter() - t0, 1)

print(f"── merged 조건 생성 ({len(eval_rows)}건)")
m_out, m_sec = generate_merged(eval_rows, desc="merged")

In [ ]:
preds = []
for o in m_out:
    obj = M.parse_json_lenient(o)
    preds.append(str(obj.get("answer", "")) if obj else o)
golds = [r.get("answer_all") or r["answer"] for r in eval_rows]

m_qa = M.score_qa(preds, golds)
m_fmt = M.format_compliance(m_out, C.REQUIRED_FIELDS, max_chars=C.MAX_OUTPUT_CHARS)

d = art.eval_dir(RUN_ID, "merged")
art.save_jsonl(d / "preds.jsonl", [
    {"id": r["id"], "question": r["question"], "gold": r["answer"],
     "raw_output": o, "parsed_answer": p} for r, o, p in zip(eval_rows, m_out, preds)])
art.save_json(d / "metrics.json", {
    "condition": "merged", "n": len(eval_rows), "em": m_qa["em"], "f1": m_qa["f1"],
    "format": m_fmt, "latency_sec_per_item": round(m_sec/len(eval_rows), 3),
    "gen_config": C.GEN_EVAL,
    "note": "4bit 베이스에서 학습한 어댑터를 16비트 베이스에 병합한 뒤 재평가"})

t = eval_summary["conditions"]["tuned"]
tbl = pd.DataFrame([
    {"조건": "tuned (4bit + 어댑터)", "EM": t["em"], "문자F1": t["f1"],
     "JSON(관대)%": t["format"]["lenient_json_pct"],
     "필수필드%": t["format"]["required_fields_pct"],
     "건당지연(초)": t["latency_per_item"]},
    {"조건": "merged (16bit 병합본)", "EM": m_qa["em"], "문자F1": m_qa["f1"],
     "JSON(관대)%": m_fmt["lenient_json_pct"],
     "필수필드%": m_fmt["required_fields_pct"],
     "건당지연(초)": round(m_sec/len(eval_rows), 3)},
])
display(tbl)

delta = {k: round(float(tbl.iloc[1][k]) - float(tbl.iloc[0][k]), 2)
         for k in ["EM", "문자F1", "JSON(관대)%", "필수필드%"]}
print("\n병합에 따른 변화 (merged - tuned):")
for k, v in delta.items():
    print(f"  {k:<14}{v:+.2f}")
verdict = "열화 관측됨" if min(delta.values()) < -1.0 else "유의한 열화 관측되지 않음"
print(f"\n판정: {verdict}")
print("(둘 다 정당한 결과다. 확인했다는 사실이 산출물이다 — §3.13 주의사항)")

In [ ]:
# 회귀 세트도 병합본으로 다시 본다 — 병합이 일반 능력을 건드렸는지
rows = [{"question": it["instruction"], "system": None, "context": None,
         "answer": "", "id": it["id"]} for it in reg_items]
r_out, _ = generate_merged(rows, gen_kwargs={"max_new_tokens": 120, "do_sample": False},
                          use_system=False, desc="regression/merged")
m_reg = REG.score_set(reg_items, r_out)
REG.print_report(m_reg, title="회귀 점검 — merged")

prev = art.load_json(art.eval_dir(RUN_ID) / "regression.json", default={})
if prev.get("tuned"):
    c2 = REG.compare(prev["tuned"], m_reg)
    print(f"\ntuned {c2['base_pct']}% -> merged {c2['tuned_pct']}%  ({c2['delta_pct']:+.1f}%p)")
    if c2["newly_broken"]:
        print("★ 병합 후 새로 깨진 문항:", c2["newly_broken"])
art.save_json(art.eval_dir(RUN_ID, "merged") / "regression.json", m_reg)

---
## 4. 병합본 저장 (선택)

수 GB이므로 Colab 디스크·Drive 용량을 확인하고 실행하십시오.
어댑터만 제출하는 것으로도 이 과정의 산출물 요건은 충족된다.

In [ ]:
SAVE_MERGED = False   # 용량을 확인한 뒤 True로

if SAVE_MERGED:
    out = art.merged_dir(RUN_ID) / "model"
    merged.save_pretrained(str(out), safe_serialization=True)
    tok.save_pretrained(str(out))
    gb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**3
    print(f"저장: {out}  ({gb:.2f} GB)")
else:
    print("병합본 저장을 건너뜁니다. 어댑터(수십 MB)만으로 재현 가능합니다.")
    print("배포용 양자화(GGUF·AWQ)와 vLLM 서빙은 후속 과정 소관입니다 — §3.13-4.")

---
## 5. 모델 카드 자동 생성 (§3.13-3)

**용도·한계·데이터 출처·실패 사례**를 문서화한다. 이것이 없으면 6개월 뒤에
이 어댑터가 무엇인지 아무도 모른다.

라이선스에 주의하십시오 — **베이스 모델의 라이선스가 파생물(어댑터·병합본)에
전파된다**(2일차 §3.6-4).

In [ ]:
# 실패 사례 3건을 직접 고르십시오 (§3.13-3의 필수 항목)
worst = sorted(zip(eval_rows, m_out, m_qa["f1_raw"]), key=lambda x: x[2])[:6]
print("F1이 가장 낮은 6건 — 이 중 3건을 골라 FAILURE_CASES에 적으십시오.\n")
for r, o, f1 in worst:
    print(f"[F1 {f1:.2f}] {r['question'][:60]}")
    print(f"    정답: {r['answer'][:80]}")
    print(f"    출력: {o[:140]!r}\n")

In [ ]:
FAILURE_CASES = [
    # {"유형": "지문 밖 생성", "예시": "...", "원인 추정": "...", "완화": "..."},
    # {"유형": "형식 붕괴", "예시": "...", "원인 추정": "...", "완화": "..."},
    # {"유형": "회귀 퇴화", "예시": "sf-04 존재하지 않는 논문에 답함", "원인 추정": "...", "완화": "..."},
]

INTENDED_USE = "사내 문서 기반 질의응답의 구조화 출력(JSON) 초안 생성"
OUT_OF_SCOPE = [
    "지문 없이 지식만으로 답해야 하는 질의 (지식 주입 목적의 파인튜닝이 아님)",
    "의료·법률·금융 등 고위험 판단",
    "사실 검증이 필요한 최종 산출물 (사람 검수 전제)",
]
BASE_LICENSE = "(베이스 모델 카드에서 확인해 기재 — 파생물에 전파된다)"
DATA_LICENSE = "(학습 데이터 라이선스 기재)"

print(f"실패 사례 {len(FAILURE_CASES)}건 작성됨 (필수 3건)")
if len(FAILURE_CASES) < 3:
    print("[주의] 3건 미만입니다. 모델 카드가 불완전합니다.")

In [ ]:
def build_model_card() -> str:
    lora = run_cfg["lora"]; tr = run_cfg["train"]; res = run_cfg["result"]
    diff = eval_summary["diff_ci"]
    lines = [
        f"# 모델 카드 — {RUN_ID}",
        "",
        "> 「LLM 작동 원리와 오픈웨이트 모델 파인튜닝」 실습 산출물. 교육 목적.",
        "",
        "## 1. 개요",
        "",
        "| 항목 | 값 |",
        "|---|---|",
        f"| 베이스 모델 | `{run_cfg['model_id']}` |",
        f"| 적응 방식 | QLoRA (4bit NF4 + 이중 양자화) + LoRA SFT |",
        f"| 학습 파라미터 | {lora['trainable_params']:,} ({lora['trainable_pct']}%) |",
        f"| 어댑터 크기 | {adapter_mb:.1f} MB |",
        f"| 의도된 용도 | {INTENDED_USE} |",
        f"| 베이스 라이선스 | {BASE_LICENSE} |",
        f"| 학습 데이터 라이선스 | {DATA_LICENSE} |",
        "",
        "## 2. 학습 설정 (재현용)",
        "",
        "| 항목 | 값 |",
        "|---|---|",
        f"| r / alpha / dropout | {lora['r']} / {lora['alpha']} / {lora['dropout']} |",
        f"| target_modules | `{lora['target_modules']}` -> 실제 적용: {', '.join(lora['modules_hit'])} |",
        f"| max_seq_len | {run_cfg['max_seq_len']} (출처: {run_cfg['max_seq_len_source']}) |",
        f"| 유효 배치 | {tr['batch']} x {tr['grad_accum']} = {tr['effective_batch']} |",
        f"| LR / 스케줄 / warmup | {tr['lr']} / {tr['scheduler']} / {tr['warmup_ratio']} |",
        f"| epochs | {tr['epochs']} |",
        f"| dtype | fp16={tr['fp16']} bf16={tr['bf16']} |",
        f"| optimizer | {tr['optim']} |",
        f"| seed | {run_cfg['seed']} |",
        f"| 학습 시간 / 피크 VRAM | {res['elapsed_min']}분 / {res['peak_vram_gb']} GB |",
        f"| GPU | {run_cfg['env'].get('gpu_name')} |",
        f"| 라이브러리 | " + ", ".join(f"{k}={v}" for k, v in run_cfg["versions"].items()) + " |",
        "",
        "### 설정 근거",
        "",
    ]
    for k, v in run_cfg.get("rationale", {}).items():
        lines.append(f"- **{k}**: {v}")
    lines += [
        "",
        "## 3. 평가",
        "",
        f"동일 프로토콜: 표본 {eval_summary['protocol']['n_eval']}건 · "
        f"`do_sample=False` · seed {eval_summary['protocol']['seed']} · "
        f"train/eval 누수 검증 {eval_summary['protocol'].get('leakage_verified')}",
        "",
        "| 지표 | base | tuned | merged |",
        "|---|---|---|---|",
        f"| EM | {eval_summary['conditions']['base']['em']} | "
        f"{eval_summary['conditions']['tuned']['em']} | {m_qa['em']} |",
        f"| 문자 F1 | {eval_summary['conditions']['base']['f1']} | "
        f"{eval_summary['conditions']['tuned']['f1']} | {m_qa['f1']} |",
        f"| JSON(관대) % | {eval_summary['conditions']['base']['format']['lenient_json_pct']} | "
        f"{eval_summary['conditions']['tuned']['format']['lenient_json_pct']} | {m_fmt['lenient_json_pct']} |",
        f"| 필수필드 % | {eval_summary['conditions']['base']['format']['required_fields_pct']} | "
        f"{eval_summary['conditions']['tuned']['format']['required_fields_pct']} | {m_fmt['required_fields_pct']} |",
        f"| 정상종료 % | {eval_summary['conditions']['base']['format']['clean_stop_pct']} | "
        f"{eval_summary['conditions']['tuned']['format']['clean_stop_pct']} | {m_fmt['clean_stop_pct']} |",
        "",
        "### 신뢰구간 (base -> tuned, 짝비교)",
        "",
        f"- EM 차이: {diff['em']['mean']:+.2f}%p (95% CI {diff['em']['lo']:+.2f} ~ {diff['em']['hi']:+.2f}) "
        f"— {'유의' if diff['em']['significant'] else '**개선을 확인하지 못했다**'}",
        f"- F1 차이: {diff['f1']['mean']:+.2f}%p (95% CI {diff['f1']['lo']:+.2f} ~ {diff['f1']['hi']:+.2f}) "
        f"— {'유의' if diff['f1']['significant'] else '**개선을 확인하지 못했다**'}",
        "",
        "### 회귀 점검",
        "",
        f"- base {eval_summary['regression']['base_pct']}% -> tuned "
        f"{eval_summary['regression']['tuned_pct']}% ({eval_summary['regression']['delta_pct']:+.1f}%p)",
        f"- 판정: **{eval_summary['regression']['verdict']}**",
        f"- 새로 깨진 문항: {eval_summary['regression']['newly_broken'] or '없음'}",
        f"- merged 회귀 통과율: {m_reg['pass_pct']}%",
        "",
        "### 병합 영향",
        "",
        "4bit 베이스에서 학습한 어댑터를 16비트 베이스에 병합한 뒤 동일 평가를 재실행했다.",
        "",
    ]
    for k, v in delta.items():
        lines.append(f"- {k}: {v:+.2f}")
    lines += [
        "",
        f"판정: **{verdict}**",
        "",
        "## 4. 한계와 금지 용도",
        "",
    ]
    for o in OUT_OF_SCOPE:
        lines.append(f"- {o}")
    lines += [
        "",
        "## 5. 실패 사례",
        "",
    ]
    if FAILURE_CASES:
        lines += ["| 유형 | 예시 | 원인 추정 | 완화 |", "|---|---|---|---|"]
        for f in FAILURE_CASES:
            lines.append(f"| {f.get('유형','')} | {f.get('예시','')} | "
                         f"{f.get('원인 추정','')} | {f.get('완화','')} |")
    else:
        lines.append("**(미작성 — 3건 필수)**")
    lines += [
        "",
        "## 6. 재현 방법",
        "",
        "```",
        "410_SFT_Dataset_Build.ipynb   -> artifacts/data/",
        f"420_QLoRA_SFT_Training.ipynb  -> artifacts/runs/{RUN_ID}/",
        f"430_Evaluate_Before_After.ipynb -> artifacts/eval/{RUN_ID}/",
        "440_Merge_And_ModelCard.ipynb -> 이 문서",
        "```",
        "",
        "위 설정표의 값을 `common/config.py`에 넣고 순서대로 실행하면 재현된다.",
        "",
        "## 7. 다음 단계",
        "",
        "배포용 양자화(GGUF·AWQ)·vLLM 서빙·처리량 튜닝·온프레미스 보안은",
        "「sLLM 양자화와 온프레미스 서빙」 과정 소관이다. 이 문서와 어댑터가 그 과정의 입력이 된다.",
    ]
    return "\n".join(lines)

card = build_model_card()
p = art.merged_dir(RUN_ID) / "model_card.md"
p.write_text(card, encoding="utf-8")
print("저장:", p)
print("\n" + "=" * 70)
print(card[:2600])

---
## 정리

### 제출
- `artifacts/merged/<run_id>/model_card.md` — **실패 사례 3건과 라이선스가 채워진 것**
- `artifacts/eval/<run_id>/merged/` — 병합본 재평가 결과

### 확인 질문
1. 어댑터만 저장하는 방식이 유리한 운영 상황을 하나 쓰시오.
2. 4bit 학습 → 16비트 병합의 불일치가 왜 생기는지 한 문장으로 쓰시오.
3. 병합 후 열화가 관측되지 않았다면, 그 사실을 모델 카드에 어떻게 써야 하는가?